# 覓聲 MeetSum — 把摘要微調版放進 Ollama
1. 從雲端硬碟取回 LoRA → 2. 下載原始權重 → 3. 把 LoRA 加回原始權重 → 4. 匯入 Ollama（模型名稱 `meetsum-summary`）
→ 5. 用覓聲正式的呼叫方式（/api/chat、format=json）比較原版與微調版 → 6. 存回雲端硬碟。原版 qwen3.8:27b 不動。

In [ ]:
# 0. 環境：程式與測試資料都內嵌在 notebook 裡；安裝並啟動 Ollama，背景下載 qwen3.8:27b
import os, sys, json, gzip, base64, subprocess, shutil, time, glob, hashlib
W = '/content/deploy'; os.makedirs(W, exist_ok=True)
for n, s in {'merge_lora.py': '"""把 LoRA 直接加回原始權重檔（逐個 safetensors 分片處理）。\n\n不經過 transformers 的 save_pretrained：輸出的檔名、分片、張量名稱、dtype 和官方 Qwen/Qwen3.8-27B 完全一樣，\n只有被 LoRA 改到的 400 個權重數值不同。這樣 Ollama / llama.cpp 讀到的就是「官方格式」，避免轉換器不認得。\n\nW_new = W + (B @ A) * (alpha / r)\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport shutil\nfrom pathlib import Path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--out", required=True)\n    a = ap.parse_args()\n\n    import torch\n    from safetensors import safe_open\n    from safetensors.torch import load_file, save_file\n\n    base, out = Path(a.base), Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    cfg = json.loads((Path(a.adapter) / "adapter_config.json").read_text())\n    scale = cfg["lora_alpha"] / cfg["r"]\n    lora = load_file(str(Path(a.adapter) / "adapter_model.safetensors"))\n    mods = sorted({k[: k.index(".lora_A")] for k in lora if ".lora_A" in k})\n    print(f"LoRA：{len(mods)} 個模組，r={cfg[\'r\']}，alpha={cfg[\'lora_alpha\']}，scale={scale}", flush=True)\n\n    index = json.loads((base / "model.safetensors.index.json").read_text())\n    wmap = index["weight_map"]\n    keys = list(wmap)\n\n    def find(mod: str) -> str:\n        m = mod.removeprefix("base_model.model.")\n        if m + ".weight" in wmap:\n            return m + ".weight"\n        tail = re.search(r"layers\\.\\d+\\..*$", m).group(0) + ".weight"\n        hits = [k for k in keys if k.endswith(tail) and "visual" not in k and "vision" not in k and "mtp" not in k]\n        if len(hits) != 1:\n            raise RuntimeError(f"{mod} 對不到唯一的原始權重：{hits[:5]}")\n        return hits[0]\n\n    target = {find(m): m for m in mods}\n    print("對應範例：", list(target.items())[:2], flush=True)\n    by_shard: dict[str, list[str]] = {}\n    for k in target:\n        by_shard.setdefault(wmap[k], []).append(k)\n\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    done = 0\n    for shard in sorted(set(wmap.values())):\n        src = base / shard\n        if shard not in by_shard:\n            shutil.copy(src, out / shard)\n            continue\n        with safe_open(str(src), "pt") as f:\n            meta = f.metadata()\n        t = load_file(str(src))\n        for k in by_shard[shard]:\n            m = target[k]\n            A = lora[m + ".lora_A.weight"].to(dev, torch.float32)\n            B = lora[m + ".lora_B.weight"].to(dev, torch.float32)\n            w = t[k]\n            delta = (B @ A) * scale\n            if delta.shape != w.shape:\n                raise RuntimeError(f"{k} 形狀不合：{tuple(w.shape)} vs {tuple(delta.shape)}")\n            t[k] = (w.to(dev, torch.float32) + delta).to(w.dtype).cpu()\n            done += 1\n        save_file(t, str(out / shard), metadata=meta)\n        print(f"  {shard}：合併 {len(by_shard[shard])} 個權重", flush=True)\n        del t\n    for f in base.iterdir():\n        if f.is_file() and not f.name.endswith(".safetensors") and not (out / f.name).exists():\n            shutil.copy(f, out / f.name)\n    if done != len(mods):\n        raise RuntimeError(f"只合併了 {done}/{len(mods)} 個")\n    print(f"完成：合併 {done} 個權重 → {out}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'ollama_eval.py': '"""用和覓聲正式系統相同的方式呼叫 Ollama（/api/chat、format=json、temperature 0、seed 42），\n在 8 場測試會議上比較：\n- 原版 qwen3.8:27b（思考模式開啟，= 目前 .env 的 LLM_THINK=True）\n- 微調版（思考模式關閉，= 訓練時的設定）\n分數用和 Colab 評測相同的 eval_sum.score。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport requests\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport eval_sum as E  # noqa: E402\n\n\ndef call(host: str, model: str, ex: dict, think: bool, num_ctx: int) -> tuple[str, dict]:\n    payload = {\n        "model": model,\n        "messages": [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}],\n        "stream": False,\n        "format": "json",\n        "options": {"temperature": 0, "seed": 42, "num_ctx": num_ctx},\n        "think": think,\n    }\n    t0 = time.time()\n    r = requests.post(f"{host}/api/chat", json=payload, timeout=3600)\n    if r.status_code != 200:\n        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")\n    body = r.json()\n    msg = body.get("message") or {}\n    stat = {"secs": round(time.time() - t0, 1), "prompt_tokens": body.get("prompt_eval_count"),\n            "out_tokens": body.get("eval_count"), "thinking_chars": len(msg.get("thinking") or ""),\n            "done_reason": body.get("done_reason")}\n    return msg.get("content") or "", stat\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="sft_test.jsonl")\n    ap.add_argument("--host", default="http://127.0.0.1:11434")\n    ap.add_argument("--model", required=True)\n    ap.add_argument("--tag", required=True)\n    ap.add_argument("--think", choices=["true", "false"], required=True)\n    ap.add_argument("--num-ctx", type=int, default=40960)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--limit", type=int, default=0)\n    a = ap.parse_args()\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(x) for x in open(a.data, encoding="utf-8") if x.strip()]\n    if a.limit:\n        rows = rows[: a.limit]\n    pred = out / f"{a.tag}.predictions.jsonl"\n    done = {}\n    if pred.exists():\n        done = {json.loads(l)["id"]: json.loads(l) for l in pred.open(encoding="utf-8")}\n    with pred.open("a", encoding="utf-8") as f:\n        for r in rows:\n            if r["id"] in done:\n                continue\n            reply, stat = call(a.host, a.model, r, a.think == "true", a.num_ctx)\n            rec = {"id": r["id"], "reply": reply, **stat}\n            done[r["id"]] = rec\n            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n            print(f"  {a.tag} {r[\'id\']}  {stat}", flush=True)\n    per = []\n    for r in rows:\n        s = E.score(done[r["id"]]["reply"], json.loads(r["target"]), r["transcript"])\n        s.update(id=r["id"], lang=r["lang"], secs=done[r["id"]]["secs"], out_tokens=done[r["id"]]["out_tokens"])\n        per.append(s)\n    rep = {"all": E.summarize(per), "zh": E.summarize([p for p in per if p["lang"] == "zh"]),\n           "en": E.summarize([p for p in per if p["lang"] == "en"]), "per_meeting": per,\n           "avg_secs": round(sum(p["secs"] for p in per) / len(per), 1)}\n    (out / f"{a.tag}.report.json").write_text(json.dumps(rep, ensure_ascii=False, indent=1), encoding="utf-8")\n    print(a.tag, json.dumps(rep["all"], ensure_ascii=False), "平均秒數", rep["avg_secs"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'eval_sum.py': '"""摘要評測：原版（不掛 LoRA）vs 微調版（掛 LoRA），和我寫的參考摘要比較。\n\n指標（每場會議算完取平均）：\n- json_ok：輸出是合法 JSON 且結構正確\n- quote_ok：引用原文確實出現在逐字稿的比例（有沒有捏造）\n- count_diff：四類項目（報告／討論／決議／待辦）數量和參考摘要的平均差距（越小越好）\n- topic_f1：報告＋討論的議題，和參考摘要逐一配對的 F1（字元相似度 ≥ 0.5 算配對成功）\n- action_f1：待辦事項和參考摘要配對的 F1\n- text_sim：整份摘要文字（headline＋各項摘要＋待辦）和參考摘要的字元 ROUGE-L F1\n- headline_sim：一句話總結的字元 ROUGE-L F1\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport difflib\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nCATS = ["reports", "discussions", "decisions", "action_items"]\n\n\ndef parse(text: str):\n    t = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()\n    t = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", t)\n    try:\n        j = json.loads(t)\n    except Exception:\n        m = re.search(r"\\{.*\\}", t, flags=re.S)\n        if not m:\n            return None\n        try:\n            j = json.loads(m.group(0))\n        except Exception:\n            return None\n    if not isinstance(j, dict) or not isinstance(j.get("meeting_content"), dict):\n        return None\n    return j\n\n\ndef _items(j, cat):\n    v = (j or {}).get("meeting_content", {}).get(cat) or []\n    return [x for x in v if isinstance(x, dict)]\n\n\ndef lcs_f1(a: str, b: str) -> float:\n    if not a or not b:\n        return 0.0\n    a, b = a[:1500], b[:1500]\n    prev = [0] * (len(b) + 1)\n    for x in a:\n        cur = [0]\n        for j, y in enumerate(b, 1):\n            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))\n        prev = cur\n    l = prev[-1]\n    p, r = l / len(a), l / len(b)\n    return 2 * p * r / (p + r) if p + r else 0.0\n\n\ndef match_f1(pred: list[str], ref: list[str], thr=0.5) -> float:\n    if not pred and not ref:\n        return 1.0\n    if not pred or not ref:\n        return 0.0\n    used, tp = set(), 0\n    for p in pred:\n        best, bi = 0, None\n        for i, r in enumerate(ref):\n            if i in used:\n                continue\n            s = difflib.SequenceMatcher(None, p.lower(), r.lower()).ratio()\n            if s > best:\n                best, bi = s, i\n        if best >= thr:\n            used.add(bi)\n            tp += 1\n    pr, rc = tp / len(pred), tp / len(ref)\n    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n\n\ndef flat_text(j) -> str:\n    parts = [str((j or {}).get("headline") or "")]\n    for cat in CATS:\n        for it in _items(j, cat):\n            parts.append(str(it.get("topic") or ""))\n            parts.append(str(it.get("summary") or it.get("task") or ""))\n            parts += [str(k) for k in (it.get("key_points") or [])]\n    return "".join(parts)\n\n\ndef score(pred_text: str, ref: dict, transcript: dict) -> dict:\n    j = parse(pred_text)\n    if j is None:\n        return {"json_ok": 0, "quote_ok": 0, "count_diff": None, "topic_f1": 0, "action_f1": 0, "text_sim": 0, "headline_sim": 0}\n    seg = {s["segment_id"]: s["text"] for s in transcript["segments"]}\n    q_tot = q_ok = 0\n    for cat in CATS:\n        for it in _items(j, cat):\n            for e in it.get("evidence") or []:\n                if not isinstance(e, dict):\n                    continue\n                q_tot += 1\n                q_ok += bool(e.get("quote")) and str(e.get("quote")) in seg.get(e.get("segment_id"), "")\n    topics = lambda x: [str(i.get("topic") or "") for c in ("reports", "discussions") for i in _items(x, c)]\n    tasks = lambda x: [str(i.get("task") or "") for i in _items(x, "action_items")]\n    return {\n        "json_ok": 1,\n        "quote_ok": q_ok / q_tot if q_tot else 0.0,\n        "count_diff": sum(abs(len(_items(j, c)) - len(_items(ref, c))) for c in CATS) / 4,\n        "topic_f1": match_f1(topics(j), topics(ref)),\n        "action_f1": match_f1(tasks(j), tasks(ref)),\n        "text_sim": lcs_f1(flat_text(j), flat_text(ref)),\n        "headline_sim": lcs_f1(str(j.get("headline") or ""), str(ref.get("headline") or "")),\n    }\n\n\ndef summarize(rows: list[dict]) -> dict:\n    out = {"n": len(rows)}\n    for k in ["json_ok", "quote_ok", "count_diff", "topic_f1", "action_f1", "text_sim", "headline_sim"]:\n        v = [r[k] for r in rows if r[k] is not None]\n        out[k] = round(sum(v) / len(v), 4) if v else None\n    return out\n\n\nclass Shared:\n    def __init__(self, base, adapter, load_4bit=False):\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        import lm_utils as L\n        self.L = L\n        self.tok = L.load_tokenizer(base)\n        self.tok.padding_side = "left"\n        self.model = L.load_model(base, adapter, load_4bit=load_4bit)\n        self.use_adapter = True\n\n    def generate(self, batch, max_new_tokens=5120):\n        import torch\n        texts = [self.L.chat_prompt(self.tok, ex["system"], ex["user"]) for ex in batch]\n        enc = self.tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)\n        ctx = contextlib.nullcontext() if self.use_adapter else self.model.disable_adapter()\n        with torch.no_grad(), ctx:\n            gen = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=self.tok.pad_token_id)\n        return self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)\n\n\ndef run(data: Path, base: str, adapter: str, out: Path, batch_size: int, load_4bit: bool):\n    out.mkdir(parents=True, exist_ok=True)\n    rows = [json.loads(x) for x in (data / "sft_test.jsonl").open(encoding="utf-8")]\n    m = Shared(base, adapter, load_4bit)\n    report = {}\n    for tag, use in [("base", False), ("ft", True)]:\n        m.use_adapter = use\n        pred_path = out / f"sum_{tag}.predictions.jsonl"\n        done = {}\n        if pred_path.exists():\n            done = {json.loads(l)["id"]: json.loads(l)["reply"] for l in pred_path.open(encoding="utf-8")}\n        todo = [r for r in rows if r["id"] not in done]\n        t0 = time.time()\n        with pred_path.open("a", encoding="utf-8") as f:\n            for i in range(0, len(todo), batch_size):\n                b = todo[i:i + batch_size]\n                for r, rep in zip(b, m.generate(b)):\n                    done[r["id"]] = rep\n                    f.write(json.dumps({"id": r["id"], "reply": rep}, ensure_ascii=False) + "\\n")\n                print(f"  {tag} {i + len(b)}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)\n        per = []\n        for r in rows:\n            s = score(done[r["id"]], json.loads(r["target"]), r["transcript"])\n            s["id"], s["lang"] = r["id"], r["lang"]\n            per.append(s)\n        report[tag] = {"all": summarize(per), "zh": summarize([p for p in per if p["lang"] == "zh"]),\n                       "en": summarize([p for p in per if p["lang"] == "en"]), "per_meeting": per}\n        (out / "sum_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")\n        print(tag, json.dumps(report[tag]["all"], ensure_ascii=False), flush=True)\n    return report\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--batch-size", type=int, default=4)\n    ap.add_argument("--load-4bit", action="store_true")\n    a = ap.parse_args()\n    run(Path(a.data), a.base, a.adapter, Path(a.out), a.batch_size, a.load_4bit)\n'}.items():
    open(f'{W}/{n}', 'w', encoding='utf-8').write(s)
open(f'{W}/test8.jsonl', 'wb').write(gzip.decompress(base64.b64decode('...')))
open(f'{W}/hf_sum_report.json', 'w', encoding='utf-8').write('{\n "base": {\n  "all": {\n   "n": 8,\n   "json_ok": 0.625,\n   "quote_ok": 0.605,\n   "count_diff": 1.8,\n   "topic_f1": 0.3086,\n   "action_f1": 0.1944,\n   "text_sim": 0.2494,\n   "headline_sim": 0.2161\n  },\n  "zh": {\n   "n": 5,\n   "json_ok": 0.4,\n   "quote_ok": 0.393,\n   "count_diff": 2.375,\n   "topic_f1": 0.1748,\n   "action_f1": 0.1111,\n   "text_sim": 0.1091,\n   "headline_sim": 0.0653\n  },\n  "en": {\n   "n": 3,\n   "json_ok": 1.0,\n   "quote_ok": 0.9583,\n   "count_diff": 1.4167,\n   "topic_f1": 0.5317,\n   "action_f1": 0.3333,\n   "text_sim": 0.4833,\n   "headline_sim": 0.4675\n  },\n  "per_meeting": [\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R003S01C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9649122807017544,\n    "count_diff": 1.5,\n    "topic_f1": 0.5882352941176471,\n    "action_f1": 0.3333333333333333,\n    "text_sim": 0.3518808777429467,\n    "headline_sim": 0.16867469879518074,\n    "id": "L_R003S02C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R004S01C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 0,\n    "quote_ok": 0,\n    "count_diff": null,\n    "topic_f1": 0,\n    "action_f1": 0,\n    "text_sim": 0,\n    "headline_sim": 0,\n    "id": "L_R004S02C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 3.25,\n    "topic_f1": 0.28571428571428575,\n    "action_f1": 0.22222222222222224,\n    "text_sim": 0.19367588932806326,\n    "headline_sim": 0.15789473684210528,\n    "id": "L_R004S03C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.0,\n    "topic_f1": 0.6666666666666666,\n    "action_f1": 0.0,\n    "text_sim": 0.5286666666666666,\n    "headline_sim": 0.5224489795918368,\n    "id": "EN2002a",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.875,\n    "count_diff": 1.5,\n    "topic_f1": 0.42857142857142855,\n    "action_f1": 0.5,\n    "text_sim": 0.4713333333333333,\n    "headline_sim": 0.430188679245283,\n    "id": "EN2002b",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.75,\n    "topic_f1": 0.5000000000000001,\n    "action_f1": 0.5,\n    "text_sim": 0.45,\n    "headline_sim": 0.4497607655502392,\n    "id": "EN2002c",\n    "lang": "en"\n   }\n  ]\n },\n "ft": {\n  "all": {\n   "n": 8,\n   "json_ok": 1.0,\n   "quote_ok": 0.9854,\n   "count_diff": 1.625,\n   "topic_f1": 0.6262,\n   "action_f1": 0.2684,\n   "text_sim": 0.4706,\n   "headline_sim": 0.4222\n  },\n  "zh": {\n   "n": 5,\n   "json_ok": 1.0,\n   "quote_ok": 0.9853,\n   "count_diff": 1.95,\n   "topic_f1": 0.6442,\n   "action_f1": 0.3495,\n   "text_sim": 0.4162,\n   "headline_sim": 0.3837\n  },\n  "en": {\n   "n": 3,\n   "json_ok": 1.0,\n   "quote_ok": 0.9855,\n   "count_diff": 1.0833,\n   "topic_f1": 0.5961,\n   "action_f1": 0.1333,\n   "text_sim": 0.5613,\n   "headline_sim": 0.4863\n  },\n  "per_meeting": [\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 2.25,\n    "topic_f1": 0.75,\n    "action_f1": 0.0,\n    "text_sim": 0.410323709536308,\n    "headline_sim": 0.35416666666666663,\n    "id": "L_R003S01C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 0.75,\n    "topic_f1": 0.823529411764706,\n    "action_f1": 0.0,\n    "text_sim": 0.5748953974895398,\n    "headline_sim": 0.4444444444444445,\n    "id": "L_R003S02C02",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9859154929577465,\n    "count_diff": 2.75,\n    "topic_f1": 0.6875,\n    "action_f1": 0.6666666666666665,\n    "text_sim": 0.42266666666666663,\n    "headline_sim": 0.3466666666666666,\n    "id": "L_R004S01C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9629629629629629,\n    "count_diff": 2.25,\n    "topic_f1": 0.39999999999999997,\n    "action_f1": 0.6363636363636364,\n    "text_sim": 0.29577956051621906,\n    "headline_sim": 0.34146341463414637,\n    "id": "L_R004S02C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9777777777777777,\n    "count_diff": 1.75,\n    "topic_f1": 0.5599999999999999,\n    "action_f1": 0.4444444444444445,\n    "text_sim": 0.3771661569826707,\n    "headline_sim": 0.4318181818181818,\n    "id": "L_R004S03C01",\n    "lang": "zh"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 0.9565217391304348,\n    "count_diff": 1.25,\n    "topic_f1": 0.6666666666666667,\n    "action_f1": 0.0,\n    "text_sim": 0.6126666666666667,\n    "headline_sim": 0.43181818181818177,\n    "id": "EN2002a",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 0.5,\n    "topic_f1": 0.5333333333333333,\n    "action_f1": 0.4,\n    "text_sim": 0.572,\n    "headline_sim": 0.555956678700361,\n    "id": "EN2002b",\n    "lang": "en"\n   },\n   {\n    "json_ok": 1,\n    "quote_ok": 1.0,\n    "count_diff": 1.5,\n    "topic_f1": 0.5882352941176471,\n    "action_f1": 0.0,\n    "text_sim": 0.49933333333333335,\n    "headline_sim": 0.47104247104247104,\n    "id": "EN2002c",\n    "lang": "en"\n   }\n  ]\n }\n}')
print('測試會議', sum(1 for _ in open(f'{W}/test8.jsonl', encoding='utf-8')), '場')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'失敗 exit {p.returncode}: {cmd}')
    return p.returncode
if shutil.which('ollama') is None:
    sh('curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1')
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'
if subprocess.run('curl -sf http://127.0.0.1:11434/api/version', shell=True).returncode != 0:
    subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
    time.sleep(5)
print(subprocess.run('ollama --version', shell=True, capture_output=True, text=True).stdout)
PULL = subprocess.Popen('ollama pull qwen3.8:27b > /content/ollama_pull.log 2>&1', shell=True)
print('qwen3.8:27b 背景下載中')

In [ ]:
# 1. 掛載雲端硬碟，取回 LoRA（20 個分片 → 組回 zip → 檢查 SHA256 → 解壓）
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/MeetSum_finetune/lora_sum'
parts = sorted(glob.glob(f'{SRC}/MeetSum_summary_ft.zip.part*'))
print(len(parts), '個分片')
with open('/content/lora.zip', 'wb') as out:
    for p in parts:
        out.write(open(p, 'rb').read())
want = open(f'{SRC}/SHA256.txt').read().split()[0]
got = hashlib.sha256(open('/content/lora.zip', 'rb').read()).hexdigest()
assert got == want, f'SHA256 不符：{got} != {want}'
sh('rm -rf /content/lora_x && mkdir -p /content/lora_x && cd /content/lora_x && unzip -q /content/lora.zip')
ADAPTER = '/content/lora_x/run/final'
print('LoRA OK：', os.listdir(ADAPTER))

In [ ]:
# 2. 下載原始權重（約 54GB）
BASE = '/content/hf/Qwen3.8-27B'
if not os.path.exists(f'{BASE}/model.safetensors.index.json'):
    from huggingface_hub import snapshot_download
    snapshot_download('Qwen/Qwen3.8-27B', local_dir=BASE, max_workers=16)
print('完成', sh(f'du -sh {BASE}', check=False))

In [ ]:
# 3. 把 LoRA 加回原始權重（檔名、張量名稱、格式和官方完全一樣）
MERGED = '/content/merged'
if not os.path.exists(f'{MERGED}/model.safetensors.index.json'):
    sh(f'{sys.executable} {W}/merge_lora.py --base {BASE} --adapter {ADAPTER} --out {MERGED}')
sh(f'du -sh {MERGED}; df -h /content | tail -1', check=False)

In [ ]:
# 4. 匯入 Ollama（模型名稱 meetsum-summary）：沿用原版的 Modelfile（對話樣板、參數），只換掉權重
NAME = 'meetsum-summary'
PULL.wait()
print('下載結果：', (open('/content/ollama_pull.log', errors='replace').read()[-300:].replace('\r', '\n').strip().splitlines() or [''])[-1], '（exit', PULL.returncode, '）')
show = subprocess.run('ollama show qwen3.8:27b', shell=True, capture_output=True, text=True).stdout
print(show)
import re
q = (re.search(r'quantization\s+(\S+)', show) or [None, 'Q4_K_M'])[1]
QUANT = {'Q4_K_M': 'q4_K_M', 'Q4_K_S': 'q4_K_S', 'Q8_0': 'q8_0'}.get(q.upper(), 'q4_K_M')
base_mf = subprocess.run('ollama show qwen3.8:27b --modelfile', shell=True, capture_output=True, text=True).stdout
def quick_test(model):
    import requests
    r = requests.post('http://127.0.0.1:11434/api/chat', json={'model': model, 'stream': False, 'format': 'json', 'think': False,
        'messages': [{'role': 'user', 'content': '用 JSON 回答：{"ok": true, "word": "測試"}'}], 'options': {'temperature': 0}}, timeout=900)
    print('快速測試', model, r.status_code, r.text[:300])
    return r.status_code == 200 and '"ok"' in r.text
def make(from_path):
    mf = re.sub(r'^FROM .*$', f'FROM {from_path}', base_mf, count=1, flags=re.M)
    mf = re.sub(r'^ADAPTER .*$\n?', '', mf, flags=re.M)
    open(f'{W}/Modelfile', 'w').write(mf)
    return sh(f'cd {W} && ollama create {NAME} --quantize {QUANT} -f Modelfile', check=False) == 0
ROUTE = None
# 做法 A：讓 Ollama 自己讀合併好的 safetensors 並量化成和原版相同的格式
if make(MERGED) and quick_test(NAME):
    ROUTE = 'A'
else:
    print('--- 做法 A 沒成功，Ollama 紀錄最後 60 行 ---')
    print(''.join(open('/content/ollama.log', errors='replace').readlines()[-60:]))
    # 做法 B：llama.cpp 轉 GGUF（bf16）→ 量化 → 匯入
    LC = '/content/llama.cpp'
    if not os.path.exists(LC):
        sh(f'git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LC}')
    sh(f'{sys.executable} -m pip install -q -r {LC}/requirements/requirements-convert_hf_to_gguf.txt', check=False)
    sh(f'cd {LC} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 12 > /dev/null')
    sh(f'{sys.executable} {LC}/convert_hf_to_gguf.py {MERGED} --outfile /content/merged-bf16.gguf --outtype bf16')
    sh(f'{LC}/build/bin/llama-quantize /content/merged-bf16.gguf /content/meetsum-summary-{q}.gguf {q.upper()}')
    os.remove('/content/merged-bf16.gguf')
    if make(f'/content/meetsum-summary-{q}.gguf') and quick_test(NAME):
        ROUTE = 'B'
if ROUTE is None:
    print(''.join(open('/content/ollama.log', errors='replace').readlines()[-80:]))
    raise RuntimeError('兩種做法都沒辦法讓 Ollama 正常執行，請看上面的紀錄')
print('✅ 匯入成功，做法', ROUTE, '，量化格式', QUANT)
sh('ollama list', check=False)

In [ ]:
# 5. 用覓聲正式的呼叫方式實測（8 場：中文 5、英文 3）
#    原版 = qwen3.8:27b + 思考模式（目前 .env 的設定）；微調版 = meetsum-summary + 不思考（訓練時的設定）
EV = f'{W}/eval'
sh(f'{sys.executable} {W}/ollama_eval.py --data {W}/test8.jsonl --model {NAME} --tag ft_ollama --think false --out {EV}')
sh('ollama stop meetsum-summary', check=False)
sh(f'{sys.executable} {W}/ollama_eval.py --data {W}/test8.jsonl --model qwen3.8:27b --tag base_ollama --think true --out {EV}')
B = json.load(open(f'{EV}/base_ollama.report.json')); F = json.load(open(f'{EV}/ft_ollama.report.json'))
HF = json.load(open(f'{W}/hf_sum_report.json'))
names = {'json_ok': 'JSON 合法', 'quote_ok': '引用正確', 'count_diff': '項目數差距（越小越好）', 'topic_f1': '議題吻合',
         'action_f1': '待辦吻合', 'text_sim': '文字相似度', 'headline_sim': '一句話總結相似度'}
for grp in ['all', 'zh', 'en']:
    print(f'\n== {grp}（{B[grp]["n"]} 場）       原版Ollama  微調Ollama  （參考：微調HF）')
    for k, label in names.items():
        print(f'{label:<14} {B[grp][k]!s:>10} {F[grp][k]!s:>10}   {HF["ft"][grp][k]!s:>8}')
print('\n平均每場秒數：原版', B['avg_secs'], '／微調版', F['avg_secs'])

In [ ]:
# 6. 存回雲端硬碟：Ollama 模型檔（GGUF）＋ Modelfile ＋ 實測結果，之後每次開 Colab 直接 ollama create 就好
DST = '/content/drive/MyDrive/MeetSum_finetune/ollama_summary'
os.makedirs(DST, exist_ok=True)
mf = subprocess.run(f'ollama show {NAME} --modelfile', shell=True, capture_output=True, text=True).stdout
blob = re.search(r'^FROM (\S+)$', mf, flags=re.M).group(1)
gguf_name = f'meetsum-summary-{QUANT}.gguf'
sh(f'df -h /content/drive | tail -1', check=False)
print('複製', blob, f'({os.path.getsize(blob) / 2**30:.1f} GB) → 雲端硬碟')
shutil.copy(blob, f'{DST}/{gguf_name}')
open(f'{DST}/Modelfile', 'w').write(re.sub(r'^FROM .*$', f'FROM ./{gguf_name}', mf, count=1, flags=re.M))
shutil.copytree(EV, f'{DST}/eval', dirs_exist_ok=True)
drive.flush_and_unmount()
print('✅ 已存到雲端硬碟 MeetSum_finetune/ollama_summary/')
print('===REPORT_BASE==='); print(json.dumps(B, ensure_ascii=False))
print('===REPORT_FT==='); print(json.dumps(F, ensure_ascii=False))
print('===END===')

In [ ]:
# 7. 關機（確認上面都完成之後才執行）
from google.colab import runtime
runtime.unassign()